# Phase 4: EDA + Choropleth Maps

**Purpose:** Explore the clean master SA2 dataset, validate stress tier assignments, and produce
publication-ready choropleth maps of water stress across South Australia.

**Inputs:**
- `data/clean/clean_master_sa2.gpkg` — 176 SA2s with geometry, SEIFA scores, water stress index and tiers

**Outputs:**
- `outputs/figures/fig_stress_tier_map.html` — choropleth of stress tiers
- `outputs/figures/fig_stress_index_map.html` — choropleth of water stress index
- `outputs/figures/fig_ier_map.html` — choropleth of IER score (economic resources)
- `outputs/figures/fig_tier_distribution.html` — bar chart of tier counts
- `outputs/figures/fig_stress_index_hist.html` — histogram of water stress index
- `outputs/figures/fig_ier_vs_stress.html` — scatter IER score vs water stress index
- `outputs/figures/fig_stress_by_tier_box.html` — box plot of stress index by tier

**Key metric:** `water_stress_index = estimated_annual_water_bill / ier_score * 1000`
IER (Index of Economic Resources) is the SEIFA proxy for household income/wealth at SA2 level.
Higher index = greater water stress relative to local economic capacity.

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent
FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print(f'Root: {ROOT}')
print(f'Figures output: {FIGURES}')

In [ ]:
# Load spatial dataset (includes geometry)
gdf = gpd.read_file(ROOT / 'data' / 'clean' / 'clean_master_sa2.gpkg')

# SA2_CODE21 must stay as string — leading zeros exist
gdf['SA2_CODE21'] = gdf['SA2_CODE21'].astype(str)

print(f'Shape: {gdf.shape}')
print(f'CRS: {gdf.crs}')
print()
print(gdf.dtypes)

In [ ]:
# Null counts — 10 SA2s have no SEIFA score (unpopulated areas, expected)
print('Null counts:')
print(gdf.isnull().sum())
print()
print('Stress tier distribution:')
print(gdf['stress_tier'].value_counts())
print()
print('Unknown (no SEIFA):', (gdf['stress_tier'] == 'Unknown').sum())

In [ ]:
# Reproject to WGS84 for Plotly (requires lat/lon)
gdf_wgs = gdf.to_crs(epsg=4326)

# Working subset: exclude the 10 SA2s with no SEIFA data for quantitative charts
gdf_scored = gdf_wgs[gdf_wgs['water_stress_index'].notna()].copy()

print(f'Total SA2s: {len(gdf_wgs)}')
print(f'Scored SA2s: {len(gdf_scored)}')
print(f'Excluded (no SEIFA): {len(gdf_wgs) - len(gdf_scored)}')

## 1. Tier Distribution

In [ ]:
tier_order = ['Critical', 'High', 'Moderate', 'Low']
tier_colours = {
    'Critical': '#d32f2f',
    'High':     '#f57c00',
    'Moderate': '#fbc02d',
    'Low':      '#388e3c',
}

tier_counts = (
    gdf_scored['stress_tier']
    .value_counts()
    .reindex(tier_order)
    .reset_index()
)
tier_counts.columns = ['stress_tier', 'count']

fig_tiers = px.bar(
    tier_counts,
    x='stress_tier', y='count',
    color='stress_tier',
    color_discrete_map=tier_colours,
    category_orders={'stress_tier': tier_order},
    title='SA2 Water Stress Tier Distribution — South Australia',
    labels={'stress_tier': 'Stress Tier', 'count': 'Number of SA2 Areas'},
    text='count',
)
fig_tiers.update_traces(textposition='outside')
fig_tiers.update_layout(showlegend=False, yaxis_range=[0, 60])
fig_tiers.show()

fig_tiers.write_html(FIGURES / 'fig_tier_distribution.html')
print('Saved fig_tier_distribution.html')

## 2. Water Stress Index Distribution

In [ ]:
# Tier boundary values for reference lines
p25 = gdf_scored['water_stress_index'].quantile(0.25)
p50 = gdf_scored['water_stress_index'].quantile(0.50)
p75 = gdf_scored['water_stress_index'].quantile(0.75)

print(f'Tier boundaries (by percentile):')
print(f'  Low / Moderate threshold (p25): {p25:.1f}')
print(f'  Moderate / High threshold (p50): {p50:.1f}')
print(f'  High / Critical threshold (p75): {p75:.1f}')
print()
print(gdf_scored['water_stress_index'].describe().round(2))

In [ ]:
fig_hist = px.histogram(
    gdf_scored,
    x='water_stress_index',
    color='stress_tier',
    color_discrete_map=tier_colours,
    category_orders={'stress_tier': tier_order},
    nbins=30,
    title='Distribution of Water Stress Index — SA2 Areas (SA)',
    labels={'water_stress_index': 'Water Stress Index (bill / IER score × 1000)', 'count': 'SA2 Count'},
)
for val, label in [(p25, 'Low|Mod'), (p50, 'Mod|High'), (p75, 'High|Crit')]:
    fig_hist.add_vline(
        x=val, line_dash='dash', line_color='black', line_width=1,
        annotation_text=f'{label}<br>{val:.0f}',
        annotation_position='top right',
    )
fig_hist.show()

fig_hist.write_html(FIGURES / 'fig_stress_index_hist.html')
print('Saved fig_stress_index_hist.html')

## 3. IER Score Distribution

In [ ]:
print('IER score stats:')
print(gdf_scored['ier_score'].describe().round(2))
print()
print('IER decile distribution:')
print(gdf_scored['ier_decile'].value_counts().sort_index())

In [ ]:
fig_ier = px.histogram(
    gdf_scored,
    x='ier_score',
    color='stress_tier',
    color_discrete_map=tier_colours,
    category_orders={'stress_tier': tier_order},
    nbins=25,
    title='IER Score Distribution by Stress Tier — SA2 Areas (SA)',
    labels={'ier_score': 'IER Score (Index of Economic Resources)', 'count': 'SA2 Count'},
)
fig_ier.show()

## 4. IER Score vs Water Stress Index

In [ ]:
fig_scatter = px.scatter(
    gdf_scored,
    x='ier_score',
    y='water_stress_index',
    color='stress_tier',
    color_discrete_map=tier_colours,
    category_orders={'stress_tier': tier_order},
    hover_name='SA2_NAME21',
    hover_data={'SA3_NAME21': True, 'ier_score': True, 'irsd_score': True,
                'estimated_annual_water_bill': ':.2f'},
    title='IER Score vs Water Stress Index — SA2 Areas (SA)',
    labels={
        'ier_score': 'IER Score (higher = more economic resources)',
        'water_stress_index': 'Water Stress Index',
    },
)
fig_scatter.update_traces(marker_size=6, marker_opacity=0.8)
fig_scatter.show()

fig_scatter.write_html(FIGURES / 'fig_ier_vs_stress.html')
print('Saved fig_ier_vs_stress.html')

## 5. Stress Index by Tier — Box Plot

In [ ]:
fig_box = px.box(
    gdf_scored,
    x='stress_tier',
    y='water_stress_index',
    color='stress_tier',
    color_discrete_map=tier_colours,
    category_orders={'stress_tier': tier_order},
    points='all',
    hover_name='SA2_NAME21',
    title='Water Stress Index by Tier — SA2 Areas (SA)',
    labels={'stress_tier': 'Stress Tier', 'water_stress_index': 'Water Stress Index'},
)
fig_box.update_traces(marker_size=4)
fig_box.update_layout(showlegend=False)
fig_box.show()

fig_box.write_html(FIGURES / 'fig_stress_by_tier_box.html')
print('Saved fig_stress_by_tier_box.html')

## 6. Top / Bottom 10 SA2s

In [ ]:
cols_display = ['SA2_NAME21', 'SA3_NAME21', 'ier_score', 'irsd_score',
                'estimated_annual_water_bill', 'water_stress_index', 'stress_tier']

print('=== TOP 10 MOST WATER-STRESSED SA2s ===')
print(gdf_scored.nlargest(10, 'water_stress_index')[cols_display].to_string(index=False))
print()
print('=== TOP 10 LEAST WATER-STRESSED SA2s ===')
print(gdf_scored.nsmallest(10, 'water_stress_index')[cols_display].to_string(index=False))

## 7. Choropleth Maps

In [ ]:
# Build GeoJSON from WGS84 reprojection for Plotly
geojson = json.loads(gdf_wgs.to_json())

# Confirm feature ID matches SA2_CODE21 string
sample_id = geojson['features'][0]['properties']['SA2_CODE21']
print(f'GeoJSON feature ID sample: {sample_id!r}  (type: {type(sample_id).__name__})')
print(f'DataFrame SA2_CODE21 sample: {gdf_wgs["SA2_CODE21"].iloc[0]!r}')

In [ ]:
# --- Map 1: Stress Tier Choropleth ---

# Include all 176 SA2s; Unknown tier (no SEIFA) rendered in grey
tier_colour_map = {
    'Critical': '#d32f2f',
    'High':     '#f57c00',
    'Moderate': '#fbc02d',
    'Low':      '#388e3c',
    'Unknown':  '#bdbdbd',
}

fig_tier_map = px.choropleth_mapbox(
    gdf_wgs,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='stress_tier',
    color_discrete_map=tier_colour_map,
    category_orders={'stress_tier': ['Critical', 'High', 'Moderate', 'Low', 'Unknown']},
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'ier_score': True,
        'water_stress_index': ':.1f',
        'estimated_annual_water_bill': ':.2f',
        'stress_tier': False,
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Water Stress Tiers — SA2 Areas, South Australia (2024-25)',
)
fig_tier_map.update_layout(
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    height=700,
)
fig_tier_map.show()

fig_tier_map.write_html(FIGURES / 'fig_stress_tier_map.html')
print('Saved fig_stress_tier_map.html')

In [ ]:
# --- Map 2: Water Stress Index Choropleth (continuous scale) ---

fig_index_map = px.choropleth_mapbox(
    gdf_scored,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='water_stress_index',
    color_continuous_scale='RdYlGn_r',
    range_color=[gdf_scored['water_stress_index'].quantile(0.05),
                 gdf_scored['water_stress_index'].quantile(0.95)],
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'stress_tier': True,
        'ier_score': True,
        'water_stress_index': ':.1f',
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Water Stress Index — SA2 Areas, South Australia (2024-25)',
    labels={'water_stress_index': 'Stress Index'},
)
fig_index_map.update_layout(
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    height=700,
)
fig_index_map.show()

fig_index_map.write_html(FIGURES / 'fig_stress_index_map.html')
print('Saved fig_stress_index_map.html')

In [ ]:
# --- Map 3: IER Score Choropleth ---

fig_ier_map = px.choropleth_mapbox(
    gdf_scored,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='ier_score',
    color_continuous_scale='RdYlGn',
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'ier_score': True,
        'ier_decile': True,
        'stress_tier': True,
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='IER Score (Economic Resources) — SA2 Areas, South Australia',
    labels={'ier_score': 'IER Score'},
)
fig_ier_map.update_layout(
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    height=700,
)
fig_ier_map.show()

fig_ier_map.write_html(FIGURES / 'fig_ier_map.html')
print('Saved fig_ier_map.html')

## 8. Tier Review

The tiers were assigned by **percentile rank** of `water_stress_index` (quartiles).
CLAUDE.md proposed ratio thresholds based on `annual_water_bill / median_household_income`.

Since the SEIFA SA2 file does not include median household income directly, Phase 3 used
`ier_score` as a proxy (Index of Economic Resources — captures household income, savings,
dwelling value, and rental payments at SA2 level).

Here we review whether the percentile-based tiers are geographically and economically sensible,
and whether hard thresholds should replace percentiles in Phase 5.

In [ ]:
# Tier summary statistics
tier_summary = (
    gdf_scored.groupby('stress_tier')
    .agg(
        count=('SA2_CODE21', 'count'),
        ier_score_median=('ier_score', 'median'),
        ier_score_min=('ier_score', 'min'),
        ier_score_max=('ier_score', 'max'),
        stress_index_median=('water_stress_index', 'median'),
        stress_index_min=('water_stress_index', 'min'),
        stress_index_max=('water_stress_index', 'max'),
        population_sum=('population', 'sum'),
    )
    .reindex(tier_order)
    .round(1)
)
print(tier_summary.to_string())

In [ ]:
# Population affected by tier
tier_pop = tier_summary[['count', 'population_sum']].copy()
tier_pop['pct_population'] = (tier_pop['population_sum'] / tier_pop['population_sum'].sum() * 100).round(1)
print('Population exposure by stress tier:')
print(tier_pop.to_string())

In [ ]:
# Population bar chart by tier
tier_pop_reset = tier_pop.reset_index()

fig_pop = px.bar(
    tier_pop_reset,
    x='stress_tier', y='population_sum',
    color='stress_tier',
    color_discrete_map=tier_colour_map,
    category_orders={'stress_tier': tier_order},
    title='Population Exposed by Water Stress Tier — South Australia',
    labels={'stress_tier': 'Stress Tier', 'population_sum': 'Population'},
    text='pct_population',
)
fig_pop.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_pop.update_layout(showlegend=False)
fig_pop.show()

fig_pop.write_html(FIGURES / 'fig_tier_population.html')
print('Saved fig_tier_population.html')

In [ ]:
# Critical tier SA2s — full list for review
critical_sa2s = (
    gdf_scored[gdf_scored['stress_tier'] == 'Critical']
    [['SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'ier_score', 'ier_decile',
      'irsd_score', 'irsd_decile', 'water_stress_index', 'population']]
    .sort_values('water_stress_index', ascending=False)
)
print(f'Critical tier SA2s ({len(critical_sa2s)}):')
print(critical_sa2s.to_string(index=False))

## 9. Tier Methodology Note

**Current approach (Phase 3 percentile-based):**
- Tiers are assigned by quartile of `water_stress_index`
- Guarantees exactly ~44 SA2s per tier by design
- Relative measure: "stressed compared to the rest of SA"

**CLAUDE.md proposed thresholds (absolute ratio-based):**
- Critical: `water_cost_burden_ratio > 0.04` (bill > 4% of income)
- High: `0.03–0.04`
- Moderate: `0.02–0.03`
- Low: `< 0.02`

**Recommendation for Phase 5:**
Phase 5 should add `median_household_income` from ABS Census TableBuilder or ABS Census Data
Summary (Table G02) and compute the true `water_cost_burden_ratio`. This unlocks:
1. Absolute thresholds that are interpretable to policymakers ("4% income on water")
2. Simulation engine: price rise × current income → new ratio → tier shift count
3. Comparison to national affordability benchmarks

Until then, the IER-proxy stress tiers are valid for relative ranking and choropleth maps.

In [ ]:
# IRSD (deprivation) vs stress tier — cross-tab
# IER decile 1 = least economic resources (most stressed)
cross = pd.crosstab(gdf_scored['ier_decile'], gdf_scored['stress_tier'])
cross = cross.reindex(columns=tier_order)
print('IER decile vs Stress Tier (rows=decile, cols=tier):')
print('(Decile 1 = least economic resources, Decile 10 = most)')
print(cross.to_string())

In [ ]:
# Final summary printout
print('=== PHASE 4 COMPLETE ===')
print(f'SA2s analysed: {len(gdf_scored)}')
print(f'Excluded (no SEIFA): {len(gdf_wgs) - len(gdf_scored)}')
print()
print('Figures saved to outputs/figures/:')
for f in sorted(FIGURES.glob('*.html')):
    print(f'  {f.name}')
print()
print('Next: Phase 5 — Feature Engineering')
print('  - Source ABS Census Table G02 for median_household_income per SA2')
print('  - Compute water_cost_burden_ratio = annual_water_bill / median_household_income')
print('  - Re-assign stress tiers using absolute ratio thresholds')
print('  - Compute hardship gap score')